In [ ]:
import numpy as np

import pandas as pd 
import matplotlib.pyplot as plt

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import KBinsDiscretizer

In [5]:
df = pd.read_csv('train.csv' , usecols=['Age' , 'Fare' , 'Survived'])
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [7]:
df.dropna(inplace=True)

In [9]:
df.isnull().sum()

Survived    0
Age         0
Fare        0
dtype: int64

In [10]:
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [13]:
x = df.iloc[: ,1:]
y = df.iloc[: , 0]

In [16]:
x_train ,x_test ,  y_train , y_test = train_test_split(x , y, test_size= 0.2 , random_state=42)

In [18]:
x_train.head(3)

,Age,Fare
328,31.0,20.5250
73,26.0,14.4542
253,30.0,16.1000


In [20]:
clf = DecisionTreeClassifier()
clf.fit(x_train , y_train)
y_pred = clf.predict(x_test)

In [21]:
accuracy_score(y_test , y_pred)

0.6363636363636364

In [22]:
np.mean(cross_val_score(DecisionTreeClassifier() , x , y , cv=10 , scoring= 'accuracy'))

np.float64(0.6275039123630674)

In [33]:
kbin_age = KBinsDiscretizer(n_bins= 10 , encode= 'ordinal' , strategy= 'quantile')
kbin_fare = KBinsDiscretizer(n_bins= 10 , encode= 'ordinal' , strategy= 'quantile')

In [34]:
trf = ColumnTransformer([
    ('first' , kbin_age , [0]),
    ('second' , kbin_fare , [1])
] , remainder='passthrough')

In [36]:
x_train_trf =  trf.fit_transform(x_train)
x_test_trf =  trf.transform(x_test)

c:\Users\Arjun Kaushik\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
c:\Users\Arjun Kaushik\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [37]:
trf.named_transformers_

{'first': KBinsDiscretizer(encode='ordinal', n_bins=10),
 'second': KBinsDiscretizer(encode='ordinal', n_bins=10)}

In [38]:
trf.named_transformers_['first'].bin_edges_

array([array([ 0.42, 14.  , 19.  , 22.  , 25.  , 28.5 , 32.  , 36.  , 42.  ,
              50.  , 80.  ])                                                ],
      dtype=object)

In [40]:
output = pd.DataFrame({
    'age' : x_train['Age'],
    'age_trf': x_train_trf[:,0],
    'fare':x_train['Fare'],
    'fare_trf':x_train_trf[:,1]  
})

In [41]:
output

,age,age_trf,fare,fare_trf
328,31.0,5.0,20.5250,5.0
73,26.0,4.0,14.4542,4.0
253,30.0,5.0,16.1000,5.0
719,33.0,6.0,7.7750,1.0
666,25.0,4.0,13.0000,4.0
...,...,...,...,...
92,46.0,8.0,61.1750,8.0
134,25.0,4.0,13.0000,4.0
337,41.0,7.0,134.5000,9.0
548,33.0,6.0,20.5250,5.0


In [42]:
output['age_labels'] = pd.cut(x=x_train['Age'] , bins=trf.named_transformers_['first'].bin_edges_[0].tolist())
output['fare_labels'] = pd.cut(x=x_train['Fare'] , bins=trf.named_transformers_['second'].bin_edges_[0].tolist())

In [ ]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_labels,fare_labels
204,18.0,1.0,8.0500,2.0,"(14.0, 19.0]","(7.896, 9.225]"
856,45.0,8.0,164.8667,9.0,"(42.0, 50.0]","(82.171, 512.329]"
494,21.0,2.0,8.0500,2.0,"(19.0, 22.0]","(7.896, 9.225]"
499,24.0,3.0,7.7958,1.0,"(22.0, 25.0]","(7.75, 7.896]"
654,18.0,1.0,6.7500,0.0,"(14.0, 19.0]","(0.0, 7.75]"
